<a href="https://colab.research.google.com/github/betulcelik1/NLP_Project/blob/main/notebooks/03_DeBERTa_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU bulunamadı.")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [1]:
%pip install -q --only-binary=:all: \
    "transformers==4.57.6" \
    "tokenizers>=0.22,<0.24" \
    "sentencepiece>=0.2.1" \
    "protobuf>=5.29,<7" \
    "accelerate>=1.0,<2"

In [2]:
import torch
import transformers
import sentencepiece
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SentencePiece:", sentencepiece.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

test = tokenizer(
    "Conservative opposition",
    "Conservatives",
    truncation=True,
    max_length=128
)

print("Tokenizer:", type(tokenizer).__name__)
print("Token count:", len(test["input_ids"]))
print("Tokens:", tokenizer.convert_ids_to_tokens(test["input_ids"]))

PyTorch: 2.14.1+cu130
Transformers: 4.57.6
SentencePiece: 0.2.3
CUDA available: True
GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer: DebertaV2Tokenizer
Token count: 6
Tokens: ['[CLS]', '▁Conservative', '▁opposition', '[SEP]', '▁Conservatives', '[SEP]']


In [3]:
import os
import subprocess
import pandas as pd
from pathlib import Path

repo_dir = "/content/SemEval-2027-Task-2-DiCo-NLI"

if not os.path.isdir(os.path.join(repo_dir, ".git")):
    subprocess.run([
        "git", "clone",
        "https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git",
        repo_dir
    ], check=True)

train_path = (
    Path(repo_dir)
    / "final_data/train"
    / "dico_nli_train_track1_participant_labeled.csv"
)

train_df = pd.read_csv(train_path)

labels = [
    "BACKWARD_ENTAILMENT",
    "EQUIVALENCE",
    "FORWARD_ENTAILMENT",
    "NEGATIVE_OTHER"
]

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)

assert train_df["label_id"].notna().all()

print("Train shape:", train_df.shape)
print("\nClass distribution:")
print(train_df["label"].value_counts())

display(train_df.head(3))

Train shape: (3042, 8)

Class distribution:
label
FORWARD_ENTAILMENT     881
BACKWARD_ENTAILMENT    881
EQUIVALENCE            802
NEGATIVE_OTHER         478
Name: count, dtype: int64


,instance_id,pair_id,text1_lang,text2_lang,text1,text2,label,label_id
0,dico_phrasis_0000001__en-en__flipped,dico_phrasis_0000001,en,en,Conservative opposition,Conservatives,FORWARD_ENTAILMENT,2
1,dico_phrasis_0000001__en-en__original,dico_phrasis_0000001,en,en,Conservatives,Conservative opposition,BACKWARD_ENTAILMENT,0
2,dico_phrasis_0000003__en-en__flipped,dico_phrasis_0000003,en,en,Japanese PM Shinzo Abe,Abe,FORWARD_ENTAILMENT,2


In [4]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

def normalize_text(s):
    return " ".join(str(s).lower().split())

parent = list(range(len(train_df)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

seen_ids = {}
seen_pairs = {}

pair_keys = []

for i, row in train_df.iterrows():
    key = tuple(sorted([
        normalize_text(row["text1"]),
        normalize_text(row["text2"])
    ]))
    pair_keys.append(key)

    pid = row["pair_id"]

    if pid in seen_ids:
        union(i, seen_ids[pid])
    else:
        seen_ids[pid] = i

    if key in seen_pairs:
        union(i, seen_pairs[key])
    else:
        seen_pairs[key] = i

groups = np.array([find(i) for i in range(len(train_df))])

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(train_df, groups=groups)
)

train_part = train_df.iloc[train_idx].reset_index(drop=True)
val_part = train_df.iloc[val_idx].reset_index(drop=True)

train_keys = set(
    tuple(sorted([normalize_text(a), normalize_text(b)]))
    for a, b in zip(train_part["text1"], train_part["text2"])
)

val_keys = set(
    tuple(sorted([normalize_text(a), normalize_text(b)]))
    for a, b in zip(val_part["text1"], val_part["text2"])
)

print("Training samples:", len(train_part))
print("Validation samples:", len(val_part))

print("Overlapping pair IDs:",
      len(set(train_part["pair_id"]) & set(val_part["pair_id"])))

print("Overlapping text pairs:",
      len(train_keys & val_keys))

print("\nTraining labels:")
print(train_part["label"].value_counts())

print("\nValidation labels:")
print(val_part["label"].value_counts())

Training samples: 2434
Validation samples: 608
Overlapping pair IDs: 0
Overlapping text pairs: 0

Training labels:
label
FORWARD_ENTAILMENT     706
BACKWARD_ENTAILMENT    706
EQUIVALENCE            632
NEGATIVE_OTHER         390
Name: count, dtype: int64

Validation labels:
label
FORWARD_ENTAILMENT     175
BACKWARD_ENTAILMENT    175
EQUIVALENCE            170
NEGATIVE_OTHER          88
Name: count, dtype: int64


In [5]:
from torch.utils.data import Dataset

MAX_LENGTH = 128

class DiCoNLIDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoding = self.tokenizer(
            str(row["text1"]),
            str(row["text2"]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            key: value.squeeze(0)
            for key, value in encoding.items()
        }

        item["labels"] = torch.tensor(
            int(row["label_id"]),
            dtype=torch.long
        )

        return item


train_dataset = DiCoNLIDataset(
    train_part, tokenizer, MAX_LENGTH
)

val_dataset = DiCoNLIDataset(
    val_part, tokenizer, MAX_LENGTH
)

sample = train_dataset[0]

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Sample keys:", sample.keys())
print("Input IDs shape:", sample["input_ids"].shape)
print("Label:", sample["labels"].item())

Train dataset: 2434
Validation dataset: 608
Sample keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
Input IDs shape: torch.Size([128])
Label: 2


In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

print("Model:", model.config.model_type)
print("Number of labels:", model.config.num_labels)

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Total parameters: {total_params:,}")
print("GPU available:", torch.cuda.is_available())

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: deberta-v2
Number of labels: 4
Total parameters: 141,897,988
GPU available: True


In [7]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    _, _, macro_f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        true_labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "precision": precision,
        "recall": recall
    }

print("Evaluation metrics ready.")

Evaluation metrics ready.


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/exp06_deberta",
    num_train_epochs=3,
    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    weight_decay=0.01,
    warmup_ratio=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="weighted_f1",
    greater_is_better=True,

    fp16=True,
    logging_steps=25,

    save_total_limit=2,
    report_to="none",

    seed=42,
    data_seed=42
)

print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("FP16:", training_args.fp16)

Epochs: 3
Learning rate: 2e-05
Batch size: 8
Gradient accumulation: 2
FP16: True


In [9]:
%pip install -q --upgrade --only-binary=:all: \
    "datasets==3.6.0" \
    "huggingface-hub>=0.34,<1.0"

In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer successfully initialized.")
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))

Trainer successfully initialized.
Training samples: 2434
Validation samples: 608


In [11]:
train_result = trainer.train()

print("\nTraining completed!")
print("Training metrics:")
print(train_result.metrics)

print("\nBest checkpoint:")
print(trainer.state.best_model_checkpoint)

print("\nBest validation Weighted F1:")
print(trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Precision,Recall
1,1.181300,1.053621,0.592105,0.479893,0.547351,0.528324,0.592105
2,0.704000,0.721502,0.741776,0.706233,0.736190,0.738253,0.741776
3,0.483100,0.711664,0.751645,0.720163,0.748575,0.747777,0.751645



Training completed!
Training metrics:
{'train_runtime': 203.6033, 'train_samples_per_second': 35.864, 'train_steps_per_second': 2.254, 'total_flos': 241836475041792.0, 'train_loss': 0.867550135178244, 'epoch': 3.0}

Best checkpoint:
/content/exp06_deberta/checkpoint-459

Best validation Weighted F1:
0.7485752802900972


In [12]:
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(val_dataset)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

print("EXP-06 Classification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(4)),
        target_names=labels,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=list(range(4))))

EXP-06 Classification Report:

                     precision    recall  f1-score   support

BACKWARD_ENTAILMENT     0.8057    0.8057    0.8057       175
        EQUIVALENCE     0.7831    0.7647    0.7738       170
 FORWARD_ENTAILMENT     0.7461    0.8229    0.7826       175
     NEGATIVE_OTHER     0.5676    0.4773    0.5185        88

           accuracy                         0.7516       608
          macro avg     0.7256    0.7176    0.7202       608
       weighted avg     0.7478    0.7516    0.7486       608


Confusion Matrix:
[[141  10  12  12]
 [  9 130  23   8]
 [  5  14 144  12]
 [ 20  12  14  42]]


In [13]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

save_dir = Path(
    "/content/drive/MyDrive/NLP_Project/EXP-06_DeBERTa"
)

save_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

print("Model saved to:", save_dir)
print("Saved files:")
for file in save_dir.iterdir():
    print("-", file.name)

Mounted at /content/drive
Model saved to: /content/drive/MyDrive/NLP_Project/EXP-06_DeBERTa
Saved files:
- config.json
- model.safetensors
- training_args.bin
- tokenizer_config.json
- special_tokens_map.json
- added_tokens.json
- spm.model


In [14]:
dev_path = (
    Path(repo_dir)
    / "final_data/dev"
    / "dico_nli_dev_track1_reference.csv"
)

dev_df = pd.read_csv(dev_path)

dev_df["label_id"] = dev_df["label"].map(label2id)

assert dev_df["label_id"].notna().all()

official_dev_dataset = DiCoNLIDataset(
    dev_df,
    tokenizer,
    MAX_LENGTH
)

print("Official Dev shape:", dev_df.shape)
print("Official Dev samples:", len(official_dev_dataset))
print("\nLabel distribution:")
print(dev_df["label"].value_counts())

Official Dev shape: (660, 9)
Official Dev samples: 660

Label distribution:
label
FORWARD_ENTAILMENT     190
BACKWARD_ENTAILMENT    190
EQUIVALENCE            174
NEGATIVE_OTHER         106
Name: count, dtype: int64


In [15]:
official_predictions = trainer.predict(official_dev_dataset)

official_y_true = official_predictions.label_ids
official_y_pred = np.argmax(
    official_predictions.predictions,
    axis=1
)

print("EXP-06 Official Dev Results:")
print(official_predictions.metrics)

print("\nClassification Report:")
print(
    classification_report(
        official_y_true,
        official_y_pred,
        labels=list(range(4)),
        target_names=labels,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        official_y_true,
        official_y_pred,
        labels=list(range(4))
    )
)

EXP-06 Official Dev Results:
{'test_loss': 0.8285762667655945, 'test_accuracy': 0.7045454545454546, 'test_macro_f1': 0.6705605596127215, 'test_weighted_f1': 0.6993860558070437, 'test_precision': 0.6973831479242276, 'test_recall': 0.7045454545454546, 'test_runtime': 2.6648, 'test_samples_per_second': 247.673, 'test_steps_per_second': 15.761}

Classification Report:
                     precision    recall  f1-score   support

BACKWARD_ENTAILMENT     0.7677    0.8000    0.7835       190
        EQUIVALENCE     0.7143    0.6609    0.6866       174
 FORWARD_ENTAILMENT     0.7311    0.8158    0.7711       190
     NEGATIVE_OTHER     0.4831    0.4057    0.4410       106

           accuracy                         0.7045       660
          macro avg     0.6741    0.6706    0.6706       660
       weighted avg     0.6974    0.7045    0.6994       660


Confusion Matrix:
[[152  13  15  10]
 [ 12 115  26  21]
 [  4  16 155  15]
 [ 30  17  16  43]]


In [16]:
import json
from sklearn.metrics import classification_report

results_dir = Path(
    "/content/drive/MyDrive/NLP_Project/EXP-06_DeBERTa"
)

results_dir.mkdir(parents=True, exist_ok=True)

predictions_df = dev_df[
    ["instance_id", "pair_id", "text1", "text2", "label"]
].copy()

predictions_df["predicted_label"] = [
    id2label[int(i)] for i in official_y_pred
]

predictions_df["correct"] = (
    predictions_df["label"] == predictions_df["predicted_label"]
)

predictions_df.to_csv(
    results_dir / "official_dev_predictions.csv",
    index=False
)

report = classification_report(
    official_y_true,
    official_y_pred,
    labels=list(range(4)),
    target_names=labels,
    output_dict=True,
    zero_division=0
)

with open(
    results_dir / "official_dev_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "experiment": "EXP-06",
            "model": MODEL_NAME,
            "epochs": 3,
            "validation_weighted_f1": trainer.state.best_metric,
            "official_dev_metrics": official_predictions.metrics,
            "classification_report": report
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved predictions:", len(predictions_df))
print("Correct predictions:", predictions_df["correct"].sum())
print("Incorrect predictions:", (~predictions_df["correct"]).sum())
print("Results saved to:", results_dir)

Saved predictions: 660
Correct predictions: 465
Incorrect predictions: 195
Results saved to: /content/drive/MyDrive/NLP_Project/EXP-06_DeBERTa


In [17]:
errors_df = predictions_df[
    ~predictions_df["correct"]
].copy()

error_summary = (
    errors_df
    .groupby(["label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("Most frequent classification errors:")
display(error_summary.head(10))

print("\nNEGATIVE_OTHER misclassifications:")
display(
    errors_df[
        errors_df["label"] == "NEGATIVE_OTHER"
    ][
        ["text1", "text2", "label", "predicted_label"]
    ].head(10)
)

Most frequent classification errors:


,label,predicted_label,count
9,NEGATIVE_OTHER,BACKWARD_ENTAILMENT,30
4,EQUIVALENCE,FORWARD_ENTAILMENT,26
5,EQUIVALENCE,NEGATIVE_OTHER,21
10,NEGATIVE_OTHER,EQUIVALENCE,17
11,NEGATIVE_OTHER,FORWARD_ENTAILMENT,16
7,FORWARD_ENTAILMENT,EQUIVALENCE,16
8,FORWARD_ENTAILMENT,NEGATIVE_OTHER,15
1,BACKWARD_ENTAILMENT,FORWARD_ENTAILMENT,15
0,BACKWARD_ENTAILMENT,EQUIVALENCE,13
3,EQUIVALENCE,BACKWARD_ENTAILMENT,12



NEGATIVE_OTHER misclassifications:


,text1,text2,label,predicted_label
554,Syria opposition unity,Syria divided opposition,NEGATIVE_OTHER,EQUIVALENCE
556,to Farrow,against Farrow,NEGATIVE_OTHER,EQUIVALENCE
558,Bangladesh collapse search,Bangladesh collapse deaths,NEGATIVE_OTHER,BACKWARD_ENTAILMENT
562,A family,A naked little girl,NEGATIVE_OTHER,BACKWARD_ENTAILMENT
563,on an orange wand,from an orange bubble hoop,NEGATIVE_OTHER,BACKWARD_ENTAILMENT
565,A blonde dog,A little blonde girl,NEGATIVE_OTHER,BACKWARD_ENTAILMENT
568,Afghans,Afghan District Governor,NEGATIVE_OTHER,BACKWARD_ENTAILMENT
569,`` political '' changes,proposed cuts,NEGATIVE_OTHER,EQUIVALENCE
572,Opens,begins operation,NEGATIVE_OTHER,EQUIVALENCE
573,two people,2,NEGATIVE_OTHER,FORWARD_ENTAILMENT


In [18]:
flip_map = {
    "FORWARD_ENTAILMENT": "BACKWARD_ENTAILMENT",
    "BACKWARD_ENTAILMENT": "FORWARD_ENTAILMENT",
    "EQUIVALENCE": "EQUIVALENCE",
    "NEGATIVE_OTHER": "NEGATIVE_OTHER"
}

pair_groups = predictions_df.groupby("pair_id")

checked = 0
consistent = 0
both_correct = 0

for pair_id, group in pair_groups:
    if len(group) != 2:
        continue

    a, b = group.iloc[0], group.iloc[1]

    checked += 1

    if flip_map[a["predicted_label"]] == b["predicted_label"]:
        consistent += 1

    if a["correct"] and b["correct"]:
        both_correct += 1

print("Checked pairs:", checked)
print("Consistent pairs:", consistent)
print("Pair consistency:", round(consistent / checked, 4))
print("Both correct:", both_correct)
print("Both-correct rate:", round(both_correct / checked, 4))

Checked pairs: 277
Consistent pairs: 240
Pair consistency: 0.8664
Both correct: 196
Both-correct rate: 0.7076
